# Trends, Overlap, outliers, correlations, importance

In [ ]:
import sys, glob, warnings

warnings.filterwarnings("ignore")
sys.path.append("../../")  # lets us import helper.py
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
from helper import *

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

DATA_FILE = glob.glob("../../data/melbourne/*.csv")[0]
TARGET = "Price"
df = pd.read_csv(DATA_FILE)


df["Date"] = pd.to_datetime(
    df["Date"], dayfirst=True, errors="coerce"
)  # sale date: text - datetime
df["Postcode"] = (
    df["Postcode"].astype("Int64").astype(str)
)  # a code, not a quantity - text
LAT = next(
    c for c in df.columns if c.lower().startswith("lat")
)  # the Kaggle file spells it "Lattitude"
LON = next(c for c in df.columns if c.lower().startswith("long"))

num_cols, cat_cols = split_columns(
    df, TARGET
)  # feature lists WITHOUT the target (Date is excluded from both)
print(
    df.shape,
    "| numeric features:",
    len(num_cols),
    "| categorical features:",
    len(cat_cols),
)

## 3a) Major trends

In [ ]:
top = (
    df[num_cols]
    .corrwith(df[TARGET])
    .abs()
    .sort_values(ascending=False)
    .head(6)
    .index.tolist()
)
print("Most correlated with Price:", top)
s = df.sample(min(4000, len(df)), random_state=1)
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, col in zip(axes.ravel(), top):
    sns.regplot(
        data=s,
        x=col,
        y=TARGET,
        scatter_kws={"alpha": 0.2, "s": 8},
        line_kws={"color": "red"},
        ax=ax,
    )
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, col in zip(axes, ["Type", "Method", "Regionname"]):
    sns.boxplot(data=df, x=col, y=TARGET, ax=ax)
    ax.set_yscale("log")
    ax.tick_params(axis="x", rotation=40)
    ax.set_title(f"Price by {col} (log scale)")
plt.tight_layout()
plt.show()

## 3b) Overlapping Data

In [ ]:
sns.scatterplot(data=s, x="Distance", y=TARGET, hue="Type", alpha=0.4)
plt.yscale("log")
plt.title("Price vs Distance by Type - overlapping clouds?")
plt.show()
r2 = {}
for col in num_cols:
    d = df[[col, TARGET]].dropna()
    r2[col] = d.corr().iloc[0, 1] ** 2
display(
    pd.Series(r2)
    .sort_values(ascending=False)
    .round(3)
    .to_frame("R2 of a straight line with Price")
)

## 3c) Outliers and Noise

In [ ]:
cols = num_cols + [TARGET]
res = (
    iqr_outliers(df, cols).join(zscore_outliers(df, cols)).join(mad_outliers(df, cols))
)
display(
    res[
        [
            "n_outliers",
            "pct",
            "n_|z|>3",
            "n_MAD_outliers",
            "lower_fence",
            "upper_fence",
            "min",
            "max",
        ]
    ]
)
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
for a, col in zip(ax, [TARGET, "Landsize", "BuildingArea", "YearBuilt"]):
    sns.boxplot(y=df[col], ax=a)
    a.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
out = pd.DataFrame(
    {
        "isolation_forest": isolation_forest_flags(df, cols),
        "elliptic_envelope": elliptic_flags(df, cols),
    }
)
print(out.sum(), "\nflagged by BOTH:", int(out.all(axis=1).sum()))
display(df[out.all(axis=1)].sort_values(TARGET, ascending=False).head(10))
display(
    df.nlargest(5, "Landsize")[
        ["Suburb", "Type", "Rooms", "Price", "Landsize", "BuildingArea"]
    ]
)
display(df.nsmallest(5, "YearBuilt")[["Suburb", "Type", "Price", "YearBuilt"]])

## 3d) Correlation and Phi-K

In [ ]:
pk, method = association_matrix(
    df.drop(columns=["Address", "Date"]), interval_cols=[c for c in num_cols + [TARGET]]
)
fig, ax = plt.subplots(1, 2, figsize=(22, 9))
sns.heatmap(
    df[num_cols + [TARGET]].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    annot_kws={"size": 7},
    ax=ax[0],
)
ax[0].set_title("Pearson correlation")
sns.heatmap(pk, cmap="viridis", vmin=0, vmax=1, ax=ax[1])
ax[1].set_title(method)
plt.tight_layout()
plt.show()
display(
    pk[TARGET]
    .drop(TARGET)
    .sort_values(ascending=False)
    .round(3)
    .to_frame(f"{method} with {TARGET}")
)

## 3e) Feature Importance

In [ ]:
from sklearn.feature_selection import f_regression, mutual_info_regression, RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

X, y, dropped = make_model_matrix(df.dropna(subset=[TARGET]), TARGET)
print("Excluded high-cardinality columns:", dropped, "| matrix:", X.shape)
Xs = X.sample(min(8000, len(X)), random_state=42)
ys = y.loc[Xs.index]
imp = pd.DataFrame(
    {
        "F_regression": f_regression(Xs, ys)[0],
        "mutual_info": mutual_info_regression(Xs, ys, random_state=42),
        "RF_importance": RandomForestRegressor(200, random_state=42, n_jobs=-1)
        .fit(Xs, ys)
        .feature_importances_,
    },
    index=Xs.columns,
)
imp["RFE_rank"] = (
    RFE(LinearRegression(), n_features_to_select=5)
    .fit(StandardScaler().fit_transform(Xs), ys)
    .ranking_
)
imp = imp.sort_values("RF_importance", ascending=False)
display(imp.round(3).head(15))
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for a, col in zip(ax, ["F_regression", "mutual_info", "RF_importance"]):
    imp[col].sort_values().tail(10).plot.barh(ax=a, color="steelblue")
    a.set_title(col)
plt.tight_layout()
plt.show()

## 3f) Additional Notes